# 예제2: BLEU 평가 지표 이해 및 계산
## 목표: BLEU(Bilingual Evaluation Understudy) 이해 및 실습

### 학습 내용
1. BLEU 지표의 개념
2. N-gram 기반 정확도
3. BLEU 점수 계산
4. 번역 품질 평가

## STEP 1: BLEU 개념 이해

In [ ]:
print("""\n【BLEU 지표란?】

BLEU (Bilingual Evaluation Understudy)
- 기계 번역 결과를 평가하는 자동 평가 지표
- 참조 번역(Reference)과 시스템 번역(Hypothesis)을 비교
- 0~1 범위의 점수 (높을수록 좋음)

【핵심 아이디어】
- N-gram 기반: 1-gram, 2-gram, 3-gram, 4-gram 매칭
- 정밀도 중심: 시스템 번역에서 참조와 일치하는 부분의 비율
- 간단하고 빠름

【사례】
참조: "the cat is on the mat"
번역: "the cat is on a mat"

1-gram 매칭: the(O), cat(O), is(O), on(O), a(X), mat(O) = 5/6
2-gram 매칭: the-cat(O), cat-is(O), is-on(O), on-a(X), a-mat(X) = 3/5
""")

## STEP 2: BLEU 계산 함수 구현

In [ ]:
from collections import Counter
import math

def tokenize(sentence):
    """문장을 토큰으로 분리"""
    return sentence.lower().split()

def get_ngrams(tokens, n):
    """N-gram 추출"""
    ngrams = []
    for i in range(len(tokens) - n + 1):
        ngram = tuple(tokens[i:i+n])
        ngrams.append(ngram)
    return ngrams

def calculate_bleu_score(reference, hypothesis, max_n=4, smooth=False):
    """
    BLEU 점수 계산
    
    Args:
        reference: 참조 번역 (문자열)
        hypothesis: 시스템 번역 (문자열)
        max_n: 최대 N-gram (보통 4)
        smooth: 스무딩 여부
    
    Returns:
        BLEU 점수 (0~1)
    """
    
    ref_tokens = tokenize(reference)
    hyp_tokens = tokenize(hypothesis)
    
    # 길이 페널티 계산
    c = len(hyp_tokens)  # 시스템 번역 길이
    r = len(ref_tokens)  # 참조 번역 길이
    
    if c > r:
        bp = 1.0
    else:
        bp = math.exp(1 - r/c) if c > 0 else 0.0
    
    # N-gram 정확도 계산
    precisions = []
    
    for n in range(1, max_n + 1):
        if len(hyp_tokens) < n:
            precisions.append(0.0)
            continue
        
        ref_ngrams = Counter(get_ngrams(ref_tokens, n))
        hyp_ngrams = Counter(get_ngrams(hyp_tokens, n))
        
        # 공통 N-gram 개수
        matching_ngrams = 0
        for ngram in hyp_ngrams:
            matching_ngrams += min(hyp_ngrams[ngram], ref_ngrams.get(ngram, 0))
        
        precision = matching_ngrams / sum(hyp_ngrams.values())
        precisions.append(precision)
    
    # BLEU 점수 계산 (가중 기하 평균)
    if min(precisions) == 0 and not smooth:
        bleu = 0.0
    else:
        log_sum = sum(math.log(p) if p > 0 else math.log(0.0001) for p in precisions)
        bleu = bp * math.exp(log_sum / len(precisions))
    
    return bleu, precisions, bp

print("✅ BLEU 계산 함수 정의 완료")

## STEP 3: BLEU 점수 계산 예시

In [ ]:
# 예시 데이터
test_cases = [
    {
        'ref': 'the cat is on the mat',
        'hyp1': 'the cat is on the mat',  # 완벽한 번역
        'hyp2': 'the cat is on a mat',    # 거의 맞는 번역
        'hyp3': 'a cat is on the mat',    # 부분 맞는 번역
        'hyp4': 'the dog is under table'  # 잘못된 번역
    }
]

print("\n【BLEU 점수 계산 예시】\n")

for case in test_cases:
    ref = case['ref']
    print(f"참조 번역: {ref}")
    print()
    
    for i in range(1, 5):
        hyp = case[f'hyp{i}']
        bleu, precisions, bp = calculate_bleu_score(ref, hyp)
        
        print(f"  시스템{i}: {hyp}")
        print(f"    BLEU: {bleu:.4f}")
        print(f"    1-gram: {precisions[0]:.3f}, 2-gram: {precisions[1]:.3f}, " + 
              f"3-gram: {precisions[2]:.3f}, 4-gram: {precisions[3]:.3f}")
        print(f"    길이 페널티 (BP): {bp:.3f}")
        print()

## STEP 4: 다양한 번역 품질 비교

In [ ]:
import pandas as pd

# 번역 품질별 테스트
translations = [
    ('i love natural language processing',
     'i love natural language processing',
     '완벽한 번역'),
    ('i love natural language processing',
     'i love natural language',
     '부분 번역 (단어 누락)'),
    ('i love natural language processing',
     'i really love natural language processing',
     '단어 추가'),
    ('i love natural language processing',
     'i love language natural processing',
     '단어 순서 오류'),
    ('i love natural language processing',
     'i hate artificial words processing',
     '의미 오류'),
]

results = []

print("\n【다양한 번역 품질 비교】\n")

for ref, hyp, description in translations:
    bleu, _, _ = calculate_bleu_score(ref, hyp)
    results.append({
        '참조': ref[:30] + '...',
        '번역': hyp[:30] + '...',
        '설명': description,
        'BLEU': f"{bleu:.4f}"
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

## STEP 5: 문장 쌍 평가

In [ ]:
# 번역 문장 쌍
sentence_pairs = [
    ('hello world', 'hello world'),
    ('hello world', 'hello there world'),
    ('hello world', 'hi world'),
    ('good morning', 'good morning'),
    ('good morning', 'morning is good'),
]

print("\n【번역 문장 쌍 평가】\n")

bleu_scores = []
for ref, hyp in sentence_pairs:
    bleu, precisions, bp = calculate_bleu_score(ref, hyp)
    bleu_scores.append(bleu)
    print(f"참조: {ref}")
    print(f"번역: {hyp}")
    print(f"BLEU: {bleu:.4f}")
    print()

# 평균 BLEU
avg_bleu = sum(bleu_scores) / len(bleu_scores)
print(f"평균 BLEU: {avg_bleu:.4f}")

## STEP 6: BLEU 해석

In [ ]:
print("""
【BLEU 점수 해석】

BLEU 범위: 0 ~ 1

점수별 품질:
  0.00 ~ 0.10: 매우 나쁨 (거의 참조와 일치하지 않음)
  0.10 ~ 0.30: 나쁨 (일부만 참조와 일치)
  0.30 ~ 0.50: 보통 (부분적으로 참조와 일치)
  0.50 ~ 0.70: 좋음 (대부분 참조와 일치)
  0.70 ~ 0.90: 매우 좋음 (거의 완벽)
  0.90 ~ 1.00: 완벽 (참조와 동일)

【장점】
  ✅ 계산이 빠르고 간단
  ✅ 다양한 참조와 비교 가능
  ✅ 자동화된 평가

【단점】
  ❌ 의미적 유사성을 고려하지 않음
  ❌ 동의어 처리 안 함
  ❌ 다양한 표현을 낮게 평가

【개선】
  - METEOR: 동의어, 어근 고려
  - ROUGE: 리콜(Recall) 기반
  - CIDEr: 이미지 캡셔닝용
""")

## 🎯 정리
- BLEU는 N-gram 기반 자동 평가 지표
- 빠르고 간단하지만 의미적 유사성은 고려 안 함
- 번역 모델 평가에 널리 사용